In [ ]:
# CozyDay AI Prototype — portable model loader
# Weights are cached in the Hugging Face default cache (or HF_HOME if set externally).
# No machine-specific cache path is hard-coded.

import os
from pathlib import Path
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM


def load_model(model_id):
    """Download/load a Hugging Face model using a portable cache location.

    On CUDA, float16 reduces VRAM use. This is NOT GGUF quantization.
    A large model may still exceed available memory and fail.
    """
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    kwargs = {"torch_dtype": torch.float16} if torch.cuda.is_available() else {}
    model = AutoModelForCausalLM.from_pretrained(model_id, **kwargs)
    if torch.cuda.is_available():
        model = model.to('cuda')
    model.eval()
    return tokenizer, model


# CozyDay AI Prototype

## 1. Research Objective

This project evaluates 15 Hugging Face language models for a daily planning assistant integrated into the CozyDay Django application.

The AI assistant should help users:
- Summarize daily PlanItems.
- Identify high-priority tasks.
- Suggest a realistic order for completing tasks.
- Provide concise, actionable planning recommendations.

## 2. Evaluation Design

All 15 models will be evaluated using the same fixed CozyDay test inputs and expected response format.

Models will be organized into five parameter-size categories, with three models in each category.

### Evaluation Criteria

Each model will be evaluated on:

1. **Instruction Adherence (0–5):** Does the model follow the requested instructions?
2. **Output Quality (0–5):** Are its recommendations relevant, clear, and useful?
3. **Formatting Consistency (0–5):** Does the response follow the required format?
4. **Response Time (seconds):** How long does generation take?
5. **Compute Requirements:** Memory usage, hardware requirements, and loading success or failure.

Higher scores indicate better performance for the three qualitative criteria. Response time and compute requirements will be recorded separately.

## 3. Team Workflow

- Models 1–5: Lin — Research Foundation
- Models 6–10: Rishabh — Prompt Evaluation
- Models 11–15: Shivani — Local LLM Integration

The same evaluation framework and test inputs will be reused across all three stages.


## Setup (clean environment)
Install project requirements from the repository root: `python -m pip install -r requirements.txt`.
For Google Colab, install the notebook-specific packages with `%pip install torch transformers pandas` in a separate setup cell if needed. Select a GPU runtime before loading large models.

The first code cell uses the Hugging Face default cache (or an externally configured `HF_HOME`); it does **not** set a Windows-only directory. Weights and large-model caches are not committed to Git.

**Important:** The baseline CSVs for Models 4–5 currently represent **GGUF Q4_K_M / llama.cpp** tests. They are not full-precision Transformers benchmarks. Their response times include process/model startup; do not compare them directly against Transformers generation-only timings.

## File paths and evaluation schema

Open the notebook from `llm-test/`, or run it from the repository root; both are supported. The saved 15 baseline records are historical measurements. **Models 4–5 were tested with quantized GGUF + llama.cpp, not original-precision Transformers.** The opt-in Transformers runner writes new, **unscored** files and does not overwrite the historical measurements. Hardware constraints may prevent larger models from loading; record such failures rather than representing them as successful tests.


In [ ]:
# Portable results directory and consistent benchmark columns
from pathlib import Path

HERE = Path.cwd().resolve()
if (HERE / 'results' / 'research_matrix.csv').exists():
    RESULTS_DIR = HERE / 'results'
elif (HERE / 'llm-test' / 'results' / 'research_matrix.csv').exists():
    RESULTS_DIR = HERE / 'llm-test' / 'results'
else:
    raise FileNotFoundError('Run from llm-test/ or repository root with llm-test/results present.')

BASELINE_COLUMNS = [
    'Model', 'Test ID', 'Status', 'Response', 'Response Time (s)',
    'Generated Tokens', 'Reached Token Limit', 'Error',
    'Instruction Adherence', 'Output Quality', 'Formatting Consistency',
    'Total Score', 'Quantization', 'Backend', 'Prompt Mode', 'Evaluation Notes',
]
print('Reading saved data from:', RESULTS_DIR)


In [ ]:

import pandas as pd

categories = [
    "Tiny (<0.5B)",
    "Small (0.5B-1.5B)",
    "Medium (>1.5B-3B)",
    "Large (>3B-7B)",
    "Very Large (>7B)"
]

rows = []

for round_number in range(3):
    for category in categories:
        rows.append({
            "Model No.": len(rows) + 1,
            "Category": category,
            "Model Name": "",
            "Hugging Face URL": "",
            "Parameters": "",
            "License / Access": "",
            "Hardware Requirements": "",
            "Loading Instructions": "",
            "Test Owner": ["Lin", "Rishabh", "Shivani"][round_number],
            "Status": "Not tested"
        })

model_matrix = pd.DataFrame(rows)

display(model_matrix)


In [ ]:

# Stage 1: Research Matrix — Tiny Models

tiny_models = [
    (0, "HuggingFaceTB/SmolLM2-135M-Instruct", "135M"),
    (5, "HuggingFaceTB/SmolLM-135M-Instruct", "135M"),
    (10, "HuggingFaceTB/SmolLM2-360M-Instruct", "360M")
]

for index, model_id, parameters in tiny_models:
    model_matrix.loc[index, "Model Name"] = model_id
    model_matrix.loc[index, "Hugging Face URL"] = (
        "https://huggingface.co/" + model_id
    )
    model_matrix.loc[index, "Parameters"] = parameters
    model_matrix.loc[index, "Hardware Requirements"] = (
        "CPU; estimate RAM needs before testing"
    )
    model_matrix.loc[index, "Loading Instructions"] = (
        "AutoTokenizer + AutoModelForCausalLM"
    )

display(model_matrix.iloc[[0, 5, 10]])


In [ ]:

# Stage 1: Research Matrix — Small Models

small_models = [
    (1, "Qwen/Qwen2.5-0.5B-Instruct", "0.5B"),
    (6, "Qwen/Qwen3-0.6B", "0.6B"),
    (11, "Qwen/Qwen2.5-1.5B-Instruct", "1.5B")
]

for index, model_id, parameters in small_models:
    model_matrix.loc[index, "Model Name"] = model_id
    model_matrix.loc[index, "Hugging Face URL"] = (
        "https://huggingface.co/" + model_id
    )
    model_matrix.loc[index, "Parameters"] = parameters
    model_matrix.loc[index, "License / Access"] = "Apache-2.0; public"
    model_matrix.loc[index, "Hardware Requirements"] = (
        "CPU possible; RAM requirement to be benchmarked"
    )
    model_matrix.loc[index, "Loading Instructions"] = (
        "AutoTokenizer + AutoModelForCausalLM"
    )

display(model_matrix.iloc[[1, 6, 11]])


In [ ]:

# Stage 1: Research Matrix — Medium Models

medium_models = [
    (2, "Qwen/Qwen3-1.7B", "1.7B",
     "Apache-2.0; public"),
    (7, "google/gemma-2-2b-it", "~2B",
     "Gemma license; acceptance required"),
    (12, "Qwen/Qwen2.5-3B-Instruct", "~3B",
     "Qwen Research License; public")
]

for index, model_id, parameters, license_info in medium_models:
    model_matrix.loc[index, "Model Name"] = model_id
    model_matrix.loc[index, "Hugging Face URL"] = (
        "https://huggingface.co/" + model_id
    )
    model_matrix.loc[index, "Parameters"] = parameters
    model_matrix.loc[index, "License / Access"] = license_info
    model_matrix.loc[index, "Hardware Requirements"] = (
        "Several GB RAM required; CPU may be slow"
    )
    model_matrix.loc[index, "Loading Instructions"] = (
        "AutoTokenizer + AutoModelForCausalLM; "
        "check model card and access requirements"
    )

display(model_matrix.iloc[[2, 7, 12]])


In [ ]:

# Stage 1: Research Matrix — Large Models

large_models = [
    (3, "microsoft/Phi-3.5-mini-instruct", "3.8B",
     "MIT; public"),
    (8, "Qwen/Qwen3-4B", "4B",
     "Apache-2.0; public"),
    (13, "Qwen/Qwen2.5-7B-Instruct", "7B",
     "Apache-2.0; public")
]

for index, model_id, parameters, license_info in large_models:
    model_matrix.loc[index, "Model Name"] = model_id
    model_matrix.loc[index, "Hugging Face URL"] = (
        "https://huggingface.co/" + model_id
    )
    model_matrix.loc[index, "Parameters"] = parameters
    model_matrix.loc[index, "License / Access"] = license_info
    model_matrix.loc[index, "Hardware Requirements"] = (
        "Approx. 8-14 GB FP16 weights; "
        "additional RAM needed for inference"
    )
    model_matrix.loc[index, "Loading Instructions"] = (
        "AutoTokenizer + AutoModelForCausalLM; "
        "review model-specific loading guidance"
    )

display(model_matrix.iloc[[3, 8, 13]])


In [ ]:

# Stage 1: Research Matrix — Very Large Models

very_large_models = [
    (4, "Qwen/Qwen3-8B", "8.2B", 8.2),
    (9, "mistralai/Mistral-Nemo-Instruct-2407", "12B", 12.0),
    (14, "Qwen/Qwen2.5-14B-Instruct", "14.7B", 14.7)
]

for index, model_id, parameters, billions in very_large_models:
    model_matrix.loc[index, "Model Name"] = model_id
    model_matrix.loc[index, "Hugging Face URL"] = (
        "https://huggingface.co/" + model_id
    )
    model_matrix.loc[index, "Parameters"] = parameters
    model_matrix.loc[index, "License / Access"] = "Apache-2.0; public"
    model_matrix.loc[index, "Hardware Requirements"] = (
        f"FP16 weights ~{billions * 2:.1f} GB; "
        "additional RAM needed; check feasibility before loading"
    )
    model_matrix.loc[index, "Loading Instructions"] = (
        "AutoTokenizer + AutoModelForCausalLM; "
        "follow model-specific chat template"
    )

display(model_matrix.iloc[[4, 9, 14]])


In [ ]:

# Complete missing license information for Tiny models

tiny_indices = [0, 5, 10]

model_matrix.loc[
    tiny_indices, "License / Access"
] = "Apache-2.0; public"

# Verify all required fields
required_columns = [
    "Model Name",
    "Hugging Face URL",
    "Parameters",
    "License / Access",
    "Hardware Requirements",
    "Loading Instructions"
]

missing = model_matrix[required_columns].isna() | (
    model_matrix[required_columns].astype(str).apply(
        lambda col: col.str.strip().eq("")
    )
)

print("Total models:", len(model_matrix))
print("Missing required fields:", int(missing.sum().sum()))
print("Unique models:", model_matrix["Model Name"].nunique())

display(model_matrix.iloc[[0, 5, 10]])


In [ ]:

# Stage 1: Fixed CozyDay Test Inputs
# These inputs must remain unchanged across all 15 models.

test_cases = [
    {
        "test_id": "T1",
        "name": "Simple Priority Planning",
        "input": (
            "I need to finish a report due tomorrow, "
            "buy groceries, and clean my room. "
            "Which task should I prioritize and why?"
        )
    },
    {
        "test_id": "T2",
        "name": "Deadline and Time Management",
        "input": (
            "I have a meeting from 2 PM to 3 PM, "
            "an assignment due at 5 PM that needs 2 hours, "
            "and a 30-minute workout. "
            "Suggest a realistic schedule starting at 12 PM."
        )
    },
    {
        "test_id": "T3",
        "name": "Busy Day Planning",
        "input": (
            "Today I need to study for 3 hours, "
            "prepare a presentation for 2 hours, "
            "reply to emails for 30 minutes, "
            "and attend class from 1 PM to 3 PM. "
            "I am available from 9 AM to 6 PM. "
            "Help me plan my day with short breaks."
        )
    }
]

for case in test_cases:
    print(f"\n{case['test_id']}: {case['name']}")
    print(case["input"])



## 4. Scoring Rubric

All models will be evaluated on the same three CozyDay test cases (T1, T2, T3).

### Required Response Format

Each model should produce:

**Priority:** The most urgent task or goal.

**Schedule:** A realistic ordered plan with time slots when appropriate.

**Reason:** A short explanation of the recommended order.

The response should be concise, actionable, and avoid inventing deadlines or task constraints.

### Scoring Dimensions

Each dimension is scored from 0 to 5.

| Score | Instruction Adherence | Output Quality | Formatting Consistency |
|---|---|---|---|
| 5 | Follows all instructions | Accurate, realistic, actionable | All three sections present and clear |
| 4 | Minor instruction omission | Useful with minor issues | Minor formatting deviations |
| 3 | Follows some instructions | Partially useful or incomplete | Two sections correctly structured |
| 2 | Follows few instructions | Major omissions or planning errors | One section correctly structured |
| 1 | Barely follows instructions | Mostly irrelevant or impractical | Unstructured response |
| 0 | Does not follow instructions | No useful answer | No requested structure |

### Overall Score

Total Score = Instruction Adherence + Output Quality + Formatting Consistency

Maximum score per test case: 15 points.

Maximum score across three test cases: 45 points.

### Additional Metrics

- **Response Time:** Seconds required for generation.
- **Compute Requirements:** Device, estimated RAM usage, and loading success/failure.
- **Error Notes:** Model access restrictions, out-of-memory errors, timeouts, or other failures.

Models that cannot load or generate output will be marked as **Not evaluated** for qualitative scores. Resource failures must be documented rather than assigned an artificial score of zero.


## Stage 1 — Manual scoring audit (official FP16 Transformers reruns)

The original rubric gives **0–5 each** for Instruction Adherence (IA), Output Quality (OQ), and Formatting Consistency (FC); each test is out of **15 points**. Generation speed and hardware are recorded separately, not scored. Raw responses remain unchanged.

| Model | Case | IA | OQ | FC | Total | Evidence |
|---|---|---:|---:|---:|---:|---|
| Phi-3.5-mini | T1 | 4 | 4 | 5 | **13** | Correct task priority; assumes report needs 2–3 hours without user-supplied estimate; all headings clear |
| Phi-3.5-mini | T2 | 2 | 2 | 4 | **8** | Fixed 2–3 PM meeting reduced to 2:30–3; extra assignment time; explanation truncated at 256 tokens |
| Phi-3.5-mini | T3 | 2 | 2 | 4 | **8** | Study 3h45m rather than 3h; 15m break mislabeled as 30m; explanation truncated at 256 tokens |
| Qwen3-8B | T1 | 5 | 5 | 5 | **15** | Appropriate report-first priority with grocery/cleaning follow-up, all three headings present |
| Qwen3-8B | T2 | 2 | 2 | 5 | **9** | Only 1h assignment work + 30m finalization versus requested 2h; redundant preparation slot; format intact |
| Qwen3-8B | T3 | 3 | 2 | 5 | **10** | Explicit study 2h versus 3h, presentation 1h45m versus 2h; unallocated time is ambiguous; format intact |

**Totals:** Phi-3.5-mini **29/45**, Qwen3-8B **34/45**. Judgments apply only to the recorded rerun outputs, not the earlier GGUF outputs. Qwen3-8B used `enable_thinking=False`; its GPU/CPU offloading latency must not be interpreted as a model-only speed comparison.


In [ ]:

# Stage 1: Shared Baseline Prompt
# Reuse this exact instruction for all 15 models.

SYSTEM_INSTRUCTION = (
    "You are CozyDay, a helpful daily planning assistant. "
    "Use only the information provided by the user. "
    "Give realistic, concise, actionable planning advice. "
    "Do not invent deadlines or appointments. "
    "Use exactly these three headings:\n"
    "Priority:\n"
    "Schedule:\n"
    "Reason:"
)

def build_prompt(tokenizer, user_input):
    """
    Format a CozyDay request using the model's chat template
    when available, or a plain-text fallback.
    """
    messages = [
        {"role": "system", "content": SYSTEM_INSTRUCTION},
        {"role": "user", "content": user_input}
    ]

    if getattr(tokenizer, "chat_template", None):
        try:
            return tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )
        except (ValueError, TypeError):
            # Some models do not support a system role.
            pass

    return (
        f"Instructions:\n{SYSTEM_INSTRUCTION}\n\n"
        f"User request:\n{user_input}\n\n"
        "Assistant response:\n"
    )

print("Baseline prompt is ready.")


In [ ]:
# Stage 1: Improved Baseline Test Harness
# Shared configuration for all 15 models

MAX_NEW_TOKENS = 256

def test_model_v2(model_id, test_cases):
    import time
    import gc
    import torch
    import pandas as pd

    results = []
    tokenizer = None
    model = None

    try:
        tokenizer, model = load_model(model_id)
        model.eval()

        for case in test_cases:
            try:
                prompt = build_prompt(tokenizer, case["input"])
                inputs = tokenizer(
                    prompt,
                    return_tensors="pt",
                    truncation=True,
                    max_length=1024
                )

                inputs = {k: v.to(next(model.parameters()).device) for k, v in inputs.items()}

                start = time.perf_counter()

                with torch.inference_mode():
                    output = model.generate(
                        **inputs,
                        max_new_tokens=MAX_NEW_TOKENS,
                        do_sample=False,
                        pad_token_id=tokenizer.eos_token_id
                    )

                elapsed = time.perf_counter() - start
                new_tokens = output[0][
                    inputs["input_ids"].shape[1]:
                ]
                response = tokenizer.decode(
                    new_tokens,
                    skip_special_tokens=True
                )

                results.append({
                    "Model": model_id,
                    "Test ID": case["test_id"],
                    "Status": "Success",
                    "Response": response,
                    "Response Time (s)": round(elapsed, 2),
                    "Generated Tokens": len(new_tokens),
                    "Reached Token Limit": (
                        len(new_tokens) >= MAX_NEW_TOKENS
                    ),
                    "Error": ""
                })

            except Exception as error:
                results.append({
                    "Model": model_id,
                    "Test ID": case["test_id"],
                    "Status": "Failed",
                    "Response": "",
                    "Response Time (s)": None,
                    "Generated Tokens": None,
                    "Reached Token Limit": None,
                    "Error": str(error)
                })

    except Exception as error:
        for case in test_cases:
            results.append({
                "Model": model_id,
                "Test ID": case["test_id"],
                "Status": "Load failed",
                "Response": "",
                "Response Time (s)": None,
                "Generated Tokens": None,
                "Reached Token Limit": None,
                "Error": str(error)
            })

    finally:
        del model
        del tokenizer
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return pd.DataFrame(results)

print("Improved baseline harness ready.")
print("Generation limit:", MAX_NEW_TOKENS)

In [ ]:
# Explicit reproducible benchmark runner for Models 1–5.
# CAUTION: This is opt-in to avoid automatically downloading multi-GB models on Run All.
# Set RUN_BASELINES = True ONLY on a machine with sufficient RAM/VRAM and disk space.
# Scores are intentionally left blank: manual scoring must be done using the shared rubric.

from pathlib import Path
import pandas as pd

MODEL_IDS_STAGE1 = {
    1: 'HuggingFaceTB/SmolLM2-135M-Instruct',
    2: 'Qwen/Qwen2.5-0.5B-Instruct',
    3: 'Qwen/Qwen3-1.7B',
    4: 'microsoft/Phi-3.5-mini-instruct',
    5: 'Qwen/Qwen3-8B',
}

RUN_BASELINES = False
MODELS_TO_RUN = [1, 2, 3, 4, 5]  # Edit to run one model at a time when necessary.

# All five model slots can be rerun independently; use the shared test cases.
# Scored official FP16 baselines must not be overwritten by reruns.
if RUN_BASELINES:
    for model_no in MODELS_TO_RUN:
        model_id = MODEL_IDS_STAGE1[model_no]
        print(f'Transformers baseline: Model {model_no} — {model_id}')
        evaluated = test_model_v2(model_id, test_cases)
        for column in ['Instruction Adherence', 'Output Quality', 'Formatting Consistency', 'Total Score']:
            evaluated[column] = pd.NA
        evaluated['Quantization'] = 'No GGUF quantization (Transformers)'
        evaluated['Backend'] = 'Transformers'
        evaluated['Prompt Mode'] = 'Baseline (SYSTEM_INSTRUCTION)'
        evaluated['Evaluation Notes'] = 'Pending manual scoring; see rubric.'
        output = RESULTS_DIR / f'model_{model_no}_transformers_rerun.csv'
        output.parent.mkdir(parents=True, exist_ok=True)
        # Never overwrite original manually scored GGUF or Transformers results.
        if output.exists():
            raise FileExistsError(f'{output} already exists; rename or archive it first.')
        evaluated = evaluated.reindex(columns=BASELINE_COLUMNS)
        evaluated.to_csv(output, index=False)
        print('Saved:', output)
else:
    print('Baselines are not automatically rerun. Set RUN_BASELINES=True to execute and save.')

In [ ]:
# Optional: reproduce the official FP16 Colab baselines for Models 4 or 5.
# Run ONLY on Colab with suitable GPU/CPU memory; set RUN_COLAB_FP16=True deliberately.
# Requires torch, transformers, accelerate. See project requirements.txt.
RUN_COLAB_FP16 = False
COLAB_MODEL_NO = 4  # set to 5 for Qwen3-8B

if RUN_COLAB_FP16:
    import torch, time, gc, pandas as pd
    from transformers import AutoTokenizer, AutoModelForCausalLM
    if not torch.cuda.is_available():
        raise RuntimeError('This reproducibility cell requires a CUDA GPU in Colab.')
    selected = MODEL_IDS_STAGE1[COLAB_MODEL_NO]
    tok = AutoTokenizer.from_pretrained(selected, trust_remote_code=False)
    kwargs = dict(dtype=torch.float16, trust_remote_code=False, low_cpu_mem_usage=True)
    if COLAB_MODEL_NO == 4:
        kwargs['attn_implementation'] = 'eager'
    elif COLAB_MODEL_NO == 5:
        kwargs.update(device_map='auto', max_memory={0:'12GiB','cpu':'7GiB'}, offload_folder='/content/qwen3_offload')
    else:
        raise ValueError('Select Model 4 or 5.')
    mdl = AutoModelForCausalLM.from_pretrained(selected, **kwargs)
    if COLAB_MODEL_NO == 4: mdl = mdl.to('cuda')
    mdl.eval()
    rows=[]
    for case in test_cases:
        opts = {'enable_thinking':False} if COLAB_MODEL_NO == 5 else {}
        msgs = [{'role':'system','content':SYSTEM_INSTRUCTION},{'role':'user','content':case['input']}]
        prompt = tok.apply_chat_template(msgs,tokenize=False,add_generation_prompt=True,**opts)
        tokens = tok(prompt,return_tensors='pt',truncation=True,max_length=1024).to('cuda')
        torch.cuda.synchronize(); t0 = time.perf_counter()
        with torch.inference_mode():
            output=mdl.generate(**tokens,max_new_tokens=256,do_sample=False,pad_token_id=tok.eos_token_id)
        torch.cuda.synchronize()
        elapsed=time.perf_counter()-t0
        new_tokens=output[0][tokens['input_ids'].shape[1]:]
        rows.append({'Model':selected,'Test ID':case['test_id'],'Status':'Success',
                     'Response':tok.decode(new_tokens,skip_special_tokens=True),
                     'Response Time (s)':round(elapsed,2),'Generated Tokens':len(new_tokens),
                     'Reached Token Limit':len(new_tokens)>=256,'Error':'',
                     'Instruction Adherence':pd.NA,'Output Quality':pd.NA,
                     'Formatting Consistency':pd.NA,'Total Score':pd.NA,
                     'Quantization':'None (FP16)',
                     'Backend':'Transformers / Colab T4 '+('GPU' if COLAB_MODEL_NO==4 else 'GPU + CPU offload'),
                     'Prompt Mode':'Baseline; thinking disabled' if COLAB_MODEL_NO==5 else 'Baseline',
                     'Evaluation Notes':'Pending manual scoring'})
        print(case['test_id'], round(elapsed,2),'seconds')
    path=RESULTS_DIR/f'model_{COLAB_MODEL_NO}_transformers_new_run.csv'
    if path.exists():raise FileExistsError(path)
    pd.DataFrame(rows,columns=BASELINE_COLUMNS).to_csv(path,index=False)
    print('Saved:',path)
    del mdl; gc.collect(); torch.cuda.empty_cache()
else:
    print('Official FP16 reproduction disabled (set RUN_COLAB_FP16=True in a suitable environment).')


In [ ]:

# Load all saved CozyDay research and baseline results
# No model downloads or inference are performed here.

from pathlib import Path
import pandas as pd

results_dir = RESULTS_DIR

model_matrix = pd.read_csv(
    results_dir / "research_matrix.csv"
)

model_results = {}

for model_no in range(1, 6):
    model_results[model_no] = pd.read_csv(
        results_dir / f"model_{model_no}_results.csv"
    )

hardware_assessment = pd.read_csv(
    results_dir / "model_4_5_hardware_assessment.csv"
)

print("Research models:", len(model_matrix))
print("Baseline models loaded:", len(model_results))
print("Baseline test records:", sum(
    len(df) for df in model_results.values()
))


In [ ]:

# Stage 1 evaluation summary: Models 1–5

summary_rows = []

for model_no, df in model_results.items():
    summary_rows.append({
        "Model": model_no,
        "Test Cases": len(df),
        "Scored Cases": int(df["Total Score"].notna().sum()),
        "Total Score / 45": df["Total Score"].sum()
    })

summary_df = pd.DataFrame(summary_rows)

display(summary_df)


## Stage 1 — Results and Team Handoff

**Completed:** 15-model research matrix; fixed test inputs; shared rubric and Transformers test harness; scored baselines for Models 1–5.

**Actual inference:** Models 1–3 were assessed with Hugging Face Transformers on local CPU. Model 4 (`microsoft/Phi-3.5-mini-instruct`) was rerun in Google Colab using the official **FP16** model and Transformers 4.57.6 on Tesla T4 GPU. Model 5 (`Qwen/Qwen3-8B`) was rerun with the official **FP16** model and Transformers 4.57.6 on Tesla T4 GPU **with CPU offloading**. No GGUF results are used as the current Model 4–5 baseline.

**Scoring review:** Model 4–5 scores were audited case by case against the shared 0–5 × 3 rubric; reasons appear above and in each CSV evaluation note. Raw Colab outputs are retained as local evidence outside the Git repository.

**Timing:** The two Colab runs report synchronized generation time, not download/model loading. CPU offloading for Model 5 produces dramatically different latency. Models 1–3 used a different execution environment. Do not interpret speed as a controlled direct comparison.

**Method details:** Both reruns used the three unchanged T1–T3 inputs, the shared `SYSTEM_INSTRUCTION`, `max_new_tokens=256`, `do_sample=False`, and the official chat templates. For Qwen3-8B, `enable_thinking=False` was explicitly set; this should be stated when comparing to earlier Qwen3 tests.

**Known limitations:** Qwen3-1.7B baseline can include `<think>` text and reach the generation limit. Model 4 T2/T3 hit 256 tokens. Model 5 T2/T3 contain scheduling errors. Save raw responses without editing them. The optional baseline runner does **not** rerun large models automatically.

**Next stage:** Rishabh handles model slots 6–10 and zero/one/few/many-shot prompting. Shivani handles 11–15 and local integration. Never commit weights, secrets or caches.
